# B7: Budget limits with alerts and cut-off
Team **collections**: budget 20,000 tokens per hour (demo window; production = monthly).
At 80% the PrometheusRule `TeamBudget80` fires to the alert receiver; at 100% MaaS cuts the team off with HTTP 429.
Open the receiver page side by side: `RECEIVER_URL`. Note: the budget resets hourly.

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
print(os.environ["RECEIVER_URL"])
import b7_budget
b7_budget.main()

**Reading the output** (one line per call while the team spends its budget):

* `HTTP 200  +N tokens  ->  total / 20,000 (x%)`: a served request, its token count, and the running total against the hourly budget.
* `client tokens sent: … waiting for TeamBudget80`: the 80% threshold was passed; now waiting for the alert rule to evaluate and Alertmanager to deliver it.
* `TeamBudget80 received after Ns`: the warning arrived at the team's inbox (open `RECEIVER_URL` to see it); `platform ratio` is the budget use as the platform measures it.
* `HTTP 429`: the budget is used up and MaaS cuts the team off.
* `cut-off: first HTTP 429 after N client tokens`: where the hard stop happened (slightly above 20,000, because requests in flight finish). `retry after cut-off -> 429` shows that further calls stay blocked.
* `TeamBudget100 received Ns after cut-off`: the critical alert.

If the very first call returns 429, this budget was already used in the last hour: create a fresh one with `scripts/b7-new-budget.sh <suffix>`.